# SHORT MAKER V2 - Vocabulary Grid Style

Tao video Short day tu vung tieng Anh theo dang poster luoi 4x4 (16 tu + hinh minh hoa).

### Quy trinh:
1. Chay **Cell 1** (cai dat, 1 lan duy nhat)
2. Nhap cau hinh o **Cell 2**
3. Chay **Cell 3** -> Video tu dong tao xong
4. Chay **Cell 4** -> Tai video ve may

In [ ]:
# @title CELL 1: CAI DAT (chay 1 lan, ~3 phut)
import os, subprocess, sys

print('Dang cai dat thu vien...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'qwen-tts', 'huggingface_hub', 'pydub', 'openai-whisper',
    'pysrt', 'requests', 'playwright', 'nest_asyncio'], check=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

os.system('apt-get install -y -qq ffmpeg sox libsox-fmt-all 2>/dev/null')
os.system('playwright install chromium 2>/dev/null')
os.system('playwright install-deps 2>/dev/null')

from IPython.display import clear_output
clear_output()

import torch, soundfile, whisper, pysrt, requests, json, re, time, shutil, subprocess
from pathlib import Path
from qwen_tts import Qwen3TTSModel
from playwright.sync_api import sync_playwright

print('TAT CA THU VIEN DA SAN SANG!')
print(f'   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('   -> Upload file giong mau (.wav) len cot trai')
print('   -> Chay Cell 2 de cau hinh')

In [ ]:
# @title CELL 2: CAU HINH
# @markdown ---
# @markdown ### Chu de tu vung
topic = 'Kitchen Vocabulary' # @param {type:'string'}
# @markdown > Vi du: Kitchen Vocabulary, Bedroom Items, Office Supplies
# @markdown ---
# @markdown ### API Keys
gemini_api_key = '' # @param {type:'string'}
# @markdown > Lay key Gemini mien phi: https://aistudio.google.com/app/apikey
# @markdown ---
# @markdown ### Ket noi TurboFlow
bridge_url = '' # @param {type:'string'}
# @markdown > Paste URL tunnel tu bridge_local.py. **Bat buoc** cho V2.
# @markdown ---
# @markdown ### Giong doc (Voice Clone)
file_giong_mau = 'yo.wav' # @param {type:'string'}
# @markdown > Upload file giong mau (.wav) len Colab (cot trai), nhap ten file.
loi_thoai_giong_mau = 'Wait, have you ever noticed that time feels faster as we get older? It is kind of scary, right? But actually, there is a hidden logic behind it.' # @param {type:'string'}
# @markdown > Noi dung ma nguoi trong file mau dang noi.

assert topic.strip(), 'Nhap chu de tu vung!'
assert gemini_api_key.strip(), 'Nhap Gemini API key!'
assert bridge_url.strip(), 'Nhap Bridge URL! V2 bat buoc dung TurboFlow.'

slug = re.sub(r'[^a-z0-9]+', '_', topic.lower()).strip('_')
print(f'Cau hinh OK! Project: {slug}')
print(f'   Chu de: {topic}')
print(f'   Giong mau: {file_giong_mau}')
print(f'   -> Chay Cell 3 de tao video')

In [ ]:
# @title CELL 3: TAO VIDEO (chay 1 lan, ~5-10 phut)
import gc, base64, threading
from IPython.display import Audio, display, HTML
from pydub import AudioSegment

P = Path(f'/content/{slug}')
P.mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f'VOCAB GRID MAKER: {topic}')
print(f"{'='*60}")

print('\n[1/5] Sinh danh sach tu vung (Gemini AI)...')

PROMPT = 'You are an expert English vocabulary educator. Create a vocabulary list for the topic: "{topic}"\n'
PROMPT += '\n'
PROMPT += 'REQUIREMENTS:\n'
PROMPT += '- Generate exactly 16 vocabulary words related to this topic.\n'
PROMPT += '- Each word: short clear English description (1 sentence, 10-15 words max).\n'
PROMPT += '- Description: simple for English learners (A2-B1 level).\n'
PROMPT += '- Also generate ONE image generation prompt for a 4x4 vocabulary grid poster.\n'
PROMPT += '\n'
PROMPT += 'VOICE SCRIPT: voice reads the word twice. Word. Word.\n'
PROMPT += 'IMAGE PROMPT: This prompt will be fed to an AI image generator. It MUST explicitly list all 16 generated vocabulary words so the image generator knows what to draw. Describe a 9:16 vertical educational poster with a 4x4 grid layout. List the 16 items for the 16 cells. Each cell has a cute illustration of the item and its name written below it.\n'
PROMPT += '\n'
PROMPT += 'OUTPUT (valid JSON only): {{"title": "...", "words": [{{"word": "...", "description": "..."}}], "grid_prompt": "...", "intro_line": "Let us learn [topic]!", "outro_line": "Follow for more!"}}\n'

messages = [{'role': 'system', 'content': 'Return valid JSON only. Generate exactly 16 vocabulary words with descriptions.'},
            {'role': 'user', 'content': PROMPT.format(topic=topic)}]

max_retries = 3
vocab_data = {}
for attempt in range(max_retries):
    try:
        gemini_contents = []
        system_instruction = None
        for msg in messages:
            if msg['role'] == 'system':
                system_instruction = {'parts': [{'text': msg['content']}]}
            else:
                role = 'user' if msg['role'] == 'user' else 'model'
                gemini_contents.append({'role': role, 'parts': [{'text': msg['content']}]})
        body = {'contents': gemini_contents, 'generationConfig': {'responseMimeType': 'application/json', 'temperature': 0.7}}
        if system_instruction:
            body['systemInstruction'] = system_instruction
        resp = requests.post(f'https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent?key={gemini_api_key.strip()}',
            headers={'Content-Type': 'application/json'}, json=body, timeout=30)
        resp.raise_for_status()
        raw = resp.json()['candidates'][0]['content']['parts'][0]['text'].strip()
        raw = re.sub(r'^```json\\s*', '', raw)
        raw = re.sub(r'\\s*```$', '', raw)
        vocab_data = json.loads(raw)
        words = vocab_data.get('words', [])
        print(f'   Da tao {len(words)} tu vung (Attempt {attempt+1}/{max_retries})')
        if len(words) == 16:
            print('   OK! Dung 16 tu!')
            break
        else:
            if attempt < max_retries - 1:
                print(f'   Co {len(words)} tu thay vi 16. Yeu cau lai...')
                messages.append({'role': 'model', 'content': raw})
                messages.append({'role': 'user', 'content': f'You generated {len(words)} words but I need exactly 16.'})
    except Exception as e:
        if attempt >= max_retries - 1: raise
        print(f'   Attempt {attempt+1} failed: {e}. Retrying...')
        time.sleep(5)

words = vocab_data.get('words', [])[:16]
grid_prompt = vocab_data.get('grid_prompt', '')
intro_line = vocab_data.get('intro_line', f'Let us learn {topic}!')
outro_line = vocab_data.get('outro_line', f'Now you know sixteen {topic.lower()} words!')
title = vocab_data.get('title', topic)
(P / 'vocab_data.json').write_text(json.dumps(vocab_data, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'   {len(words)} tu vung, prompt anh luoi da san sang')
for i, w in enumerate(words, 1): print(f'      {i:2d}. {w["word"]}')

print('\n[2/5] Tao anh luoi...')
burl = bridge_url.strip().rstrip('/')
print(f'   Ket noi TurboFlow: {burl}')
health_resp = requests.get(f'{burl}/health', timeout=5)
assert health_resp.ok, 'Khong ket noi duoc bridge!'
server_time = health_resp.json().get('time', time.time())
print('   Bridge connected!')
(P / 'prompts.txt').write_text(grid_prompt, encoding='utf-8')
Path('/content/prompts.txt').write_text(grid_prompt, encoding='utf-8')
print(f'   Da tao prompts.txt (1 prompt duy nhat)')
print(f'   Prompt: {grid_prompt[:120]}...')
print('\n   Hay tai file /content/prompts.txt tu thu muc ben trai Colab ve may.')
print('   1. Mo Edge -> Google Flow -> mo extension TurboFlow.')
print('   2. Copy noi dung prompts.txt, dan vao o Prompts.')
print('   3. Nhap Start -> cho tao xong 1 anh.')
print('   Dang cho anh tai ve thu muc local...')

since_ts = server_time - 14400
grid_image_item = None
for poll in range(600):
    time.sleep(2)
    try:
        imgs_resp = requests.get(f'{burl}/images', params={'since': since_ts}, timeout=10)
        if imgs_resp.ok:
            all_items = imgs_resp.json().get('items', [])
            if all_items:
                all_items.sort(key=lambda x: x.get('mtime', 0), reverse=True)
                grid_image_item = all_items[0]
                print(f'\n      Da phat hien anh: {grid_image_item["name"]}')
                break
            elif poll % 15 == 0:
                print(f'\r      Dang cho anh tai xuong...', end='', flush=True)
    except: pass
print()
assert grid_image_item is not None, 'Het thoi gian cho!'
grid_img_path = P / 'grid_poster.jpg'
print(f'      Dang tai: {grid_image_item["name"]} -> grid_poster.jpg')
img_data = requests.get(f'{burl}/download', params={'name': grid_image_item['name']}, timeout=30)
grid_img_path.write_bytes(img_data.content)
print('   Anh luoi da tai thanh cong!')

print(f'\n[3/5] Tao giong doc (Voice Clone tu {file_giong_mau})...')
assert os.path.exists(file_giong_mau), f'Chua thay file giong mau: {file_giong_mau}'
tts_model = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base', torch_dtype=torch.float16,
    device_map='cuda:0', attn_implementation='sdpa')
print('   Dang hoc giong mau...')
clone_prompt = tts_model.create_voice_clone_prompt(
    ref_audio=file_giong_mau, ref_text=loi_thoai_giong_mau.strip(), x_vector_only_mode=False)
nghi_ngan = AudioSegment.silent(duration=400)
nghi_dai = AudioSegment.silent(duration=700)
final_audio = AudioSegment.silent(duration=300)
voice_lines = [intro_line]
for i, w in enumerate(words, 1): voice_lines.append(f'{w["word"]}. ... {w["word"]}.')
voice_lines.append(outro_line)
word_start_times = {}
print(f'   Thu am {len(voice_lines)} cau...')
current_time_ms = 300
for li, line_text in enumerate(voice_lines):
    print(f'   [{li+1}/{len(voice_lines)}]: {line_text[:70]}...')
    with torch.inference_mode():
        w_audio, sr = tts_model.generate_voice_clone(text=line_text, voice_clone_prompt=clone_prompt)
    soundfile.write('/content/temp_line.wav', w_audio[0], sr)
    seg = AudioSegment.from_wav('/content/temp_line.wav')
    if 1 <= li <= 16: word_start_times[li - 1] = current_time_ms / 1000.0
    final_audio += seg + nghi_ngan
    current_time_ms += len(seg) + 400
    if li == 0 or li == len(voice_lines) - 1:
        final_audio += nghi_dai
        current_time_ms += 700
    if os.path.exists('/content/temp_line.wav'): os.remove('/content/temp_line.wav')
mp3_path = str(P / 'audio.mp3')
final_audio.export(mp3_path, format='mp3')
audio_duration_ms = len(final_audio)
del tts_model, clone_prompt; gc.collect(); torch.cuda.empty_cache()
print(f'   audio.mp3 da tao ({audio_duration_ms/1000:.1f}s)')
display(Audio(mp3_path, autoplay=False))

print('\n[4/5] Tao phu de tung tu (Whisper)...')
w_model = whisper.load_model('base.en')
result = w_model.transcribe(mp3_path, word_timestamps=True, language='en')
del w_model; gc.collect(); torch.cuda.empty_cache()
word_ts = [{'word': w['word'].strip(), 'start': round(w['start'], 3), 'end': round(w['end'], 3)}
           for seg in result['segments'] for w in seg.get('words', [])]
(P / 'word_timestamps.json').write_text(json.dumps(word_ts, indent=2, ensure_ascii=False), encoding='utf-8')
subtitles = []
for seg in result['segments']:
    subtitles.append({'text': seg['text'].strip(), 'start': seg['start'], 'end': seg['end'],
        'words': [{'word': w['word'].strip(), 'start': w['start'], 'end': w['end']} for w in seg.get('words', [])]})
dur_result = subprocess.run(['ffmpeg', '-i', mp3_path], capture_output=True, text=True, errors='ignore')
dur_match = re.search(r'Duration:\\s*(\\d+):(\\d+):(\\d+\\.\\d+)', dur_result.stderr)
audio_duration = int(dur_match.group(1))*3600 + int(dur_match.group(2))*60 + float(dur_match.group(3)) if dur_match else audio_duration_ms / 1000.0
print(f'   {len(subtitles)} subtitle segments, {len(word_ts)} tu')

vocab_timeline = []
for wi, w in enumerate(words):
    target_word = w['word'].lower().strip()
    best_seg = None; best_score = 0
    for seg in result['segments']:
        seg_text = seg['text'].lower()
        if target_word in seg_text:
            if f'number {wi+1}' in seg_text or str(wi+1) in seg_text:
                best_seg = seg; best_score = 2; break
            elif best_score < 1: best_seg = seg; best_score = 1
    if best_seg:
        vocab_timeline.append({'word_index': wi, 'word': w['word'], 'start': best_seg['start'], 'end': best_seg['end']})
    elif wi in word_start_times:
        vocab_timeline.append({'word_index': wi, 'word': w['word'], 'start': word_start_times[wi], 'end': word_start_times[wi] + 4.0})
if len(vocab_timeline) < len(words):
    per_word = audio_duration / len(words)
    vocab_timeline = [{'word_index': wi, 'word': words[wi]['word'], 'start': wi*per_word, 'end': (wi+1)*per_word} for wi in range(len(words))]
(P / 'vocab_timeline.json').write_text(json.dumps(vocab_timeline, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'   Timeline: {len(vocab_timeline)} tu mapped')

print(f'\n[5/5] Render video (Playwright)...')
import base64 as b64mod
grid_bytes = grid_img_path.read_bytes()
grid_b64 = b64mod.b64encode(grid_bytes).decode('ascii')
grid_ext = 'png' if grid_img_path.suffix.lower() == '.png' else 'jpeg'
grid_data_uri = f'data:image/{grid_ext};base64,{grid_b64}'
video_data = {'title': title, 'gridImage': grid_data_uri, 'words': [w['word'] for w in words],
    'vocabTimeline': vocab_timeline, 'subtitles': subtitles, 'audioDuration': audio_duration}

# Write template HTML
template_path = P / 'template_grid.html'
import urllib.request
# Copy template from the uploaded template_grid.html or use inline
template_src = Path('/content/template_grid.html')
if not template_src.exists():
    # Fallback: read from project dir if pre-uploaded
    template_src = Path('template_grid.html')
if template_src.exists():
    shutil.copy2(str(template_src), str(template_path))
else:
    raise FileNotFoundError('Upload template_grid.html to Colab!')

fps = 30
total_frames = int(audio_duration * fps) + 1
frames_dir = P / 'frames'
if frames_dir.exists(): shutil.rmtree(str(frames_dir))
frames_dir.mkdir()

def _render_task():
    with sync_playwright() as pw:
        browser = pw.chromium.launch(headless=True, args=['--no-sandbox', '--disable-dev-shm-usage', '--ignore-gpu-blocklist', '--enable-gpu', '--use-gl=egl'])
        page = browser.new_page(viewport={'width': 1080, 'height': 1920})
        page.add_init_script(f'window.videoData = {json.dumps(video_data)};')
        page.goto(f'file:///{str(template_path.resolve())}')
        page.wait_for_function('window.isReady === true')
        page.wait_for_timeout(1000)
        print(f'   Rendering {total_frames} frames @ {fps}fps...')
        t0 = time.time()
        for fi in range(total_frames):
            t = fi / fps
            page.evaluate(f'window.renderFrame({t})')
            page.screenshot(path=str(frames_dir / f'frame_{fi:05d}.jpg'), type='jpeg', quality=90)
            if (fi + 1) % 60 == 0:
                elapsed_r = time.time() - t0
                speed = (fi + 1) / elapsed_r if elapsed_r > 0 else 0
                print(f'   {fi+1}/{total_frames} frames ({speed:.1f} fps)...')
        browser.close()
    print(f'   {total_frames} frames rendered')

render_thread = threading.Thread(target=_render_task)
render_thread.start()
render_thread.join()

print('\nEncoding video (FFmpeg)...')
output_path = P / 'output.mp4'
cmd = ['ffmpeg', '-y', '-framerate', str(fps), '-i', f'{frames_dir}/frame_%05d.jpg',
       '-i', mp3_path, '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
       '-c:a', 'aac', '-b:a', '192k', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
       str(output_path)]
enc_result = subprocess.run(cmd, capture_output=True, text=True, errors='ignore')
assert enc_result.returncode == 0, f'FFmpeg error: {enc_result.stderr[-500:]}'
shutil.rmtree(str(frames_dir), ignore_errors=True)
size_mb = output_path.stat().st_size / (1024*1024)
print(f'\n{"="*60}')
print(f'VIDEO HOAN TAT! ({size_mb:.1f} MB)')
print(f'   {output_path}')
print(f'{"="*60}')
print('-> Chay Cell 4 de tai video ve may')

In [ ]:
# @title CELL 4: TAI VIDEO VE MAY
from google.colab import files
files.download(str(output_path))
print('Dang tai video...')